In [ ]:
%load_ext autoreload
%autoreload 2

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from notebooks.summersoc.globals import PATH_METRICS_DEMO_EXPLORE, PATH_MODEL_LIST, ITERATE_THROUGH_X_PARTS, \
    SPLIT_DATA_INTO_X_PARTS, PATH_METRICS_DEMO_EXPLOIT

plt.style.use('default')

from agent.components import RASK
from agent.components.GaussianProcess import GASK
from agent.components.commons import ServiceType, FIG_SIZE_DEFAULT
from agent.components.commons import SloSet

services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SloSet.DEFAULT, SloSet.HIGH_PERF, SloSet.LOW_COST, SloSet.HIGH_QUALITY]

In [ ]:
_df_explore = pd.read_csv(PATH_METRICS_DEMO_EXPLORE)
_df_exploit = pd.read_csv(PATH_METRICS_DEMO_EXPLOIT)
df_joint = pd.concat([_df_explore, _df_exploit])
df_preprocessed = RASK.preprocess_data(df_joint)

## **Analyze**: Regression Analysis of Structural Knowledge

Use the metrics to identify the current state of the system and understand how well it performs.

![Monitoring Infrastructure](img/RASK_Methodology.jpg)

### Create Structural Knowledge

Given the potential infinite amount of metrics, a common problem is to identify metrics that have a direct impact on the system's objectives. One way to identify such factors is through structural knowledge. E.g., given that you know that your ML _model size_ impacts system performance, you can decide to monotitor _model size_ with increasing accuracy.

In many cases, you have some prior understanding (i.e., expert knowledge) of your system dynamics. However, a more robust method is directly learning such relations from the monitored data.

The below example show how to identify directed causal structures from three variables. For more complex scenarios, my colleague **Nefeli Marina Rouska** will show a poster about how to choose and configure algorithms for structural learning.

In [ ]:
from utils import visualize_DAG

from pgmpy.base import DAG
from pgmpy.causal_discovery import PC

df_filtered = df_preprocessed[df_preprocessed['service_type'] == ServiceType.QR.value]
df_filtered = df_filtered[['cores', 'throughput', 'data_quality']]

estimator = PC().fit(df_filtered)
dag: DAG = estimator.causal_graph_

regular = '#a1b2ff'  # blue
special = '#c46262'  # red

visualize_DAG(dag, color_map=[regular, special, regular])

### Functional Data Analysis

Note that both variants commonly benefit from removing outliers from the dataset; in our case, it was not needed according to our judgment.



In [ ]:
rm_list = []
gp_list = []
lml_history = []

#### Variant 1: Using a Simple Regression Model

Low complexity (TODO: give the asymptotic runtime and the actual physical runtime)

Fitting 180 regression models (3 services * 60 iterations) with up to 60 samples each  \\\
Physical runtime: 500ms

Does **not** provide any degree of uncertainty about the encoded knowledge

In [ ]:
from agent.components.RASK import RASK

rm_list = []

for i in range(ITERATE_THROUGH_X_PARTS):
    rm_all_services = {}
    data_ratio = (i + 1) / SPLIT_DATA_INTO_X_PARTS

    draw_figures = False  #i == (ITERATE_THROUGH_X_PARTS / 4) - 1 or i == ITERATE_THROUGH_X_PARTS - 1
    draw_figures = i == (ITERATE_THROUGH_X_PARTS / 4) - 1 or i == ITERATE_THROUGH_X_PARTS - 1
    _rm = RASK(show_figures=draw_figures)
    _rm.init_models(df_joint, data_density=data_ratio)
    rm_list.append(_rm)

    if (i + 1) % 10 == 0:
        print(f"Finished fitting {i + 1} / {ITERATE_THROUGH_X_PARTS} of the Regression Models")


#### Variant 2: Using a Gaussian Process

Medium complexity

Has uncertainty

In [ ]:
Fitting 180 regression models (3 services * 60 iterations) with up to 60 samples each  \\\
Physical runtime: 500ms

Can model the inherent stochasticity of the processing environment and the respective uncertainty.

In [ ]:
# TODO: Give the option to change between interactive plot and not; the static
#  plot would also always render withing the notebook.

import plotly.io as pio

pio.renderers.default = 'browser'

lml_history = []

gp_list = []
lml_history = []

for i in range(ITERATE_THROUGH_X_PARTS):
    gp_all_services = {}
    data_ratio = (i + 1) / SPLIT_DATA_INTO_X_PARTS
    lml_all_service = []

    for s in services:
        # Initialize and train GP model
        draw_figures = i == (ITERATE_THROUGH_X_PARTS / 4) - 1 or i == ITERATE_THROUGH_X_PARTS - 1
        _gp = GASK(s, create_figures=draw_figures, display_figures=draw_figures)
        _gp.init_model(df_joint, data_density=data_ratio)

        _lml = _gp.get_model_lml(s, "max_tp")
        lml_scaled = _lml / data_ratio
        lml_all_service.append(lml_scaled)
        gp_all_services[s] = _gp

    lml_history.append(lml_all_service)
    gp_list.append(gp_all_services)

    if (i + 1) % 10 == 0:
        print(f"Finished fitting {i + 1} / {ITERATE_THROUGH_X_PARTS} of the Gaussian Processes")



In [ ]:
joblib.dump({'gp_list': gp_list, 'rm_list': rm_list}, PATH_MODEL_LIST)

#### Analyzing GPs fitting

A common KPI of how well a GP can express data is the Log Marginal Likelihood (LML). In brief, if we add points to the GP and the LML increases, then we are likely creating a model that can represent the data well. Looking at the below figure, this is no guarantee, e.g., in the beginning, the models think that they can represent the data well, but when including more samples, the LML improves again.

In [ ]:
data_ratios = np.arange(1, ITERATE_THROUGH_X_PARTS + 1) / SPLIT_DATA_INTO_X_PARTS
plt.figure(figsize=FIG_SIZE_DEFAULT)

# Service labels matching your 'services' list order
labels = ['QR Detector', 'CV Analyzer', 'PC Visualizer']
colors = ['#d62728', '#1f77b4', '#2ca02c']  # Red, Blue, Green


lml_array = np.array(lml_history)

for i in range(lml_array.shape[1]):
    plt.plot(data_ratios * (SPLIT_DATA_INTO_X_PARTS / ITERATE_THROUGH_X_PARTS), lml_array[:, i],
             marker='o', markersize=4, linewidth=2,
             label=labels[i], color=colors[i])

# 3. Formatting
abs_samples = int(len(df_joint) * (ITERATE_THROUGH_X_PARTS / SPLIT_DATA_INTO_X_PARTS))
plt.xlabel(f'Ratio of Training Data ({int(abs_samples / 3)} Samples)', fontsize=12)
plt.ylabel('Scaled LML / # Samples', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title="Service Type")
plt.show()